In [ ]:
import os
import gc
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import cv2
from tqdm import tqdm
import re
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet import preprocess_input
from tensorflow.keras.models import Model, load_model 
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
import psutil

In [ ]:
model_name = "plain_model_straight_eps_24epoch"
validation_df_path = "./ordered_validation_df.csv"

training_df_path1 = "./ordered_training_df.csv"
training_df_path2 = "./ordered_augmented_df.csv"

output_folder = "./training_diff_thresholds_final"

chunk_size = 1000  # Number of rows per chunk
# chunk_size = 3
thresholds = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]  # List of thresholds to apply
lr = 0.01

In [ ]:
y_columns = ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC"]

In [ ]:
def load_model(model_name):

    if os.path.exists(f"./{model_name}.h5"):
        print(f"Loading existing model from .h5 format: {model_name}.h5")
        model = tf.keras.models.load_model(f"./{model_name}.h5")
    elif os.path.exists(f"./{model_name}.keras"):
        print(f"Loading existing model from .keras format: {model_name}.keras")
        model = tf.keras.models.load_model(f"./{model_name}.keras")
    else:
        print(f"No existing model found. Starting with a new model.")

    for layer in model.layers:
        if "mobilenetv2" in layer.name.lower():  # Ensure you're targeting the correct layers
            layer.trainable = False

    model.compile(optimizer=Adam(learning_rate=lr),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    return model



# def preprocess_chunk(chunk):
#     # Select feature columns (start with 'pixel_')
#     X = chunk.filter(regex=r'^pixel_').values
#     X = X.reshape((-1, 224, 224, 3))
#     # Select target columns (exclude those with 'image' in the name)
#     Y = chunk.loc[:, ~((chunk.columns.str.contains('image')) | (chunk.columns.str.contains('pixel_')))].values
#     return X, Y
    

def preprocess_chunk(chunk):
    # Extract pixel columns (flattened image data)
    pixel_columns = [col for col in chunk.columns if col.startswith("pixel_")]
    X = chunk[pixel_columns].values  # Extract flattened pixel data
    X = X.reshape((-1, 224, 224, 3))  # Reshape to (batch, 224, 224, 3)

    # Extract image paths (if available)
    image_paths = chunk["image_path"] if "image_path" in chunk.columns else pd.Series([""] * len(chunk))

    # Extract one-hot encoded labels
    Y = chunk[y_columns].values

    return chunk, X, Y 


def preprocess_chunk_old(chunk):
    # Select feature columns (start with 'pixel_')
    X = chunk.filter(regex=r'^pixel_').values
    X = X.reshape((-1, 224, 224, 3))
    # Select target columns (exclude those with 'image' in the name)
    Y = chunk.loc[:, ~((chunk.columns.str.contains('image')) | (chunk.columns.str.contains('pixel_')))].values
    return X, Y

In [ ]:
validation_df = pd.read_csv(validation_df_path)
val_X, val_Y = preprocess_chunk_old(validation_df)

In [ ]:
batch_size = 32
model = load_model(model_name)
val_loss, val_acc = model.evaluate(val_X, val_Y, batch_size=batch_size)
print(f"  Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_acc:.4f}")

In [ ]:
best_plain_model = load_model(model_name)

In [ ]:
# def process_chunk(model, chunk, threshold):
#     # Extract features (X), labels (Y), and retain full dataset
#     full_chunk, X, Y = preprocess_chunk(chunk)

#     # X = X.astype('float32')
#     # Y = Y.astype('float32')

#     # Predict class probabilities
#     predictions = model.predict(X, batch_size=32)

#     # Compute max and second max probabilities
#     max_probs = np.max(predictions, axis=1)
#     second_max_probs = np.sort(predictions, axis=1)[:, -2]

#     # Compute confidence difference
#     confidence_diff = max_probs - second_max_probs

#     # Determine rejection condition
#     rejection = confidence_diff < threshold
#     predicted_classes = np.argmax(predictions, axis=1)

#     # Modify Y labels to include rejection class (RJ)
#     modified_Y = np.copy(Y)  # Keep the original structure
#     RJ_column = np.zeros((Y.shape[0], 1))  # Initialize RJ column as all zeros

#     # Apply rejection logic
#     modified_Y[rejection] = 0  # Set all class values to 0 if rejected
#     RJ_column[rejection] = 1  # Set RJ to 1 for rejected examples

#     # Convert modified Y back to DataFrame
#     new_y_columns = y_columns + ["RJ"]  # Add RJ class
#     modified_Y_df = pd.DataFrame(np.hstack((modified_Y, RJ_column)), columns=new_y_columns)

#     # Add confidence difference column for testing
#     # full_chunk["confidence_diff"] = confidence_diff

#     # Replace the original labels in the full dataset with the modified ones
#     full_chunk = full_chunk.drop(columns=y_columns)  # Drop old labels
#     full_chunk = pd.concat([full_chunk, modified_Y_df], axis=1)  # Add modified labels

#     del X, Y, predictions, modified_Y, RJ_column
#     gc.collect()


#     return full_chunk  # Return the full dataset with the rejection class & confidence difference


In [ ]:
# new ver:

import pandas as pd
import numpy as np
import os
import gc

def process_chunk(model, chunk, threshold):
    """
    Process a chunk of data:
    - Extracts features and labels
    - Computes confidence difference
    - Adds a rejection column (`RJ`)
    - Ensures the original row count is maintained
    """
    # Extract features (X), labels (Y), and retain full dataset
    full_chunk, X, Y = preprocess_chunk(chunk)

    # Predict class probabilities
    predictions = model.predict(X, batch_size=32)

    # Compute max and second max probabilities
    max_probs = np.max(predictions, axis=1)
    second_max_probs = np.sort(predictions, axis=1)[:, -2]

    # Compute confidence difference
    confidence_diff = max_probs - second_max_probs

    # Determine rejection condition
    rejection = confidence_diff < threshold

    # Initialize rejection column
    RJ_column = np.zeros((Y.shape[0], 1))  
    modified_Y = np.copy(Y)  # Copy original labels

    # Apply rejection logic
    modified_Y[rejection] = 0  # Set all classes to 0 if rejected
    RJ_column[rejection] = 1  # Mark rejected rows

    # Convert modified Y back to DataFrame
    modified_Y_df = pd.DataFrame(modified_Y, columns=y_columns)
    modified_Y_df["RJ"] = RJ_column  # Add rejection column

    # Ensure row order is maintained
    full_chunk = full_chunk.reset_index(drop=True)
    modified_Y_df = modified_Y_df.reset_index(drop=True)

    # Merge the modified labels back into the original chunk
    full_chunk = pd.concat([full_chunk.drop(columns=y_columns, errors='ignore'), modified_Y_df], axis=1)

    # Verify row count integrity
    assert full_chunk.shape[0] == chunk.shape[0], "Row count mismatch after processing!"

    del X, Y, predictions, modified_Y, RJ_column
    gc.collect()

    return full_chunk  # Return the updated dataset


def process_file(input_path, output_path, model, threshold, chunk_size):
    """
    Process a CSV file in chunks and save the processed output.
    - Ensures row count integrity.
    - No duplicates.
    - RJ column is correctly added.
    """
    if not os.path.exists(output_path):
        with pd.read_csv(input_path, chunksize=chunk_size) as reader:
            with open(output_path, "w") as writer:
                total_rows = 0

                for i, chunk in enumerate(reader):
                    print(f"Processing chunk {i + 1} for threshold {threshold}")
                    processed_chunk = process_chunk(model, chunk, threshold)

                    # Verify row count integrity
                    assert processed_chunk.shape[0] == chunk.shape[0], f"Row count mismatch in chunk {i+1}!"

                    # Append to CSV
                    processed_chunk.to_csv(writer, index=False, header=(i == 0))

                    total_rows += processed_chunk.shape[0]

                    del processed_chunk
                    gc.collect()

        print(f"Threshold {threshold} processing complete. Results saved to: {output_path}")
        print(f"Total rows processed: {total_rows}")

# Processing for each threshold
for threshold in thresholds:
    print(f"Processing for threshold: {threshold}")

    # Process first dataset
    process_file(
        input_path=training_df_path1,
        output_path=os.path.join(output_folder, f"training_data_threshold_{threshold}.csv"),
        model=best_plain_model,
        threshold=threshold,
        chunk_size=chunk_size
    )

    # Process second dataset
    process_file(
        input_path=training_df_path2,
        output_path=os.path.join(output_folder, f"training_augmented_data_threshold_{threshold}.csv"),
        model=best_plain_model,
        threshold=threshold,
        chunk_size=chunk_size
    )

print("Processing complete for all thresholds.")


In [ ]:
# # Process the CSV file in chunks for each threshold
# for threshold in thresholds:    

#     print(f"Processing for threshold: {threshold}")

#     output_csv1 = os.path.join(output_folder, f"training_data_threshold_{threshold}.csv")
#     if not os.path.exists(output_csv1):

#         with pd.read_csv(training_df_path1, chunksize=chunk_size) as reader, open(output_csv1, "w") as writer:
#             for i, chunk in enumerate(reader):
#                 print(f"Processing chunk {i + 1} for threshold {threshold}")
#                 processed_chunk = process_chunk(best_plain_model, chunk, threshold)
                
#                 # Write the processed chunk to the output CSV
#                 if i == 0:
#                     processed_chunk.to_csv(writer, index=False)
#                 else:
#                     processed_chunk.to_csv(writer, index=False, header=False)

#                 del processed_chunk
#                 gc.collect()


#         print(f"Threshold {threshold} processing complete. Results saved to:", output_csv1)

            
#     output_csv2 = os.path.join(output_folder, f"training_augmented_data_threshold_{threshold}.csv")
#     if not os.path.exists(output_csv2):
    
#         with pd.read_csv(training_df_path2, chunksize=chunk_size) as reader, open(output_csv2, "w") as writer:
#             for i, chunk in enumerate(reader):
#                 print(f"Processing chunk {i + 1} for threshold {threshold}")
#                 processed_chunk = process_chunk(best_plain_model, chunk, threshold)
                
#                 # Write the processed chunk to the output CSV
#                 if i == 0:
#                     processed_chunk.to_csv(writer, index=False)
#                 else:
#                     processed_chunk.to_csv(writer, index=False, header=False)

#                 del processed_chunk
#                 gc.collect()
            
#         print(f"Threshold {threshold} processing complete. Results saved to:", output_csv2)


# print("Processing complete for all thresholds.")


In [ ]:
with pd.read_csv(f"./training_diff_thresholds/training_data_threshold_{threshold}.csv", chunksize=3) as reader:
    for i, chunk in enumerate(reader):
        print(i)
        print(chunk)
        break

In [ ]:
y_columns_2 = ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC", "RJ"]

# Initialize a dictionary to store cumulative counts
class_counts = {col: 0 for col in y_columns_2}

# Process CSV in chunks
read_chunk_size = 1000 # Adjust based on your system’s memory
for chunk in pd.read_csv(f"./training_diff_thresholds/training_augmented_data_threshold_{threshold}.csv", chunksize=read_chunk_size):
    # Sum up occurrences for each class in this chunk
    class_counts.update((col, class_counts[col] + chunk[col].sum()) for col in y_columns_2)

# Convert to a DataFrame for better visualization
class_counts_df = pd.DataFrame.from_dict(class_counts, orient="index", columns=["Count"])

# Print the final class counts
print(class_counts_df)

In [ ]:
# Define class labels
y_columns_2 = ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC", "RJ"]

# Dictionary to store final class counts per threshold
final_counts = {threshold: {col: 0 for col in y_columns_2} for threshold in thresholds}

# Process each threshold file
read_chunk_size = 1000  # Adjust based on memory

for threshold in thresholds:
    # File paths
    aug_file = f"./training_diff_thresholds/training_augmented_data_threshold_{threshold}.csv"
    orig_file = f"./training_diff_thresholds/training_data_threshold_{threshold}.csv"

    # Initialize class counts
    class_counts = {col: 0 for col in y_columns_2}

    # Read augmented data in chunks and accumulate counts
    for chunk in pd.read_csv(aug_file, chunksize=read_chunk_size):
        for col in y_columns_2:
            class_counts[col] += chunk[col].sum()

    # Read original (non-augmented) data in chunks and accumulate counts
    for chunk in pd.read_csv(orig_file, chunksize=read_chunk_size):
        for col in y_columns_2:
            class_counts[col] += chunk[col].sum()

    # Store in final dictionary
    final_counts[threshold] = class_counts

# Convert to DataFrame
final_df = pd.DataFrame.from_dict(final_counts, orient="index")
final_df.index.name = "Threshold"

# Print the final DataFrame
print(final_df)

# Optionally, save to CSV
final_df.to_csv("class_counts_by_threshold.csv")


In [ ]:
# Define class labels
y_columns_2 = ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC", "RJ"]

# Dictionary to store final class counts per threshold
final_counts = {}

# Process each threshold file
read_chunk_size = 1000  # Adjust based on memory

for threshold in thresholds:
    print(f"starting for threshold: {threshold}")
    try:
        # File paths
        aug_file = f"./training_diff_thresholds_final/training_augmented_data_threshold_{threshold}.csv"
        orig_file = f"./training_diff_thresholds_final/training_data_threshold_{threshold}.csv"

        # Initialize class counts
        class_counts = {col: 0 for col in y_columns_2}

        # Read augmented data in chunks and accumulate counts
        for chunk in pd.read_csv(aug_file, chunksize=read_chunk_size):
            for col in y_columns_2:
                class_counts[col] += chunk[col].sum()

        # Read original (non-augmented) data in chunks and accumulate counts
        for chunk in pd.read_csv(orig_file, chunksize=read_chunk_size):
            for col in y_columns_2:
                class_counts[col] += chunk[col].sum()

        # Store in final dictionary
        final_counts[threshold] = class_counts
        
        print(threshold)
        print(class_counts)
        print('------------------')

    except Exception as e:
        print(f"Skipping threshold {threshold} due to error: {e}")

# Convert to DataFrame
final_df = pd.DataFrame.from_dict(final_counts, orient="index")
final_df.index.name = "Threshold"

# Print the final DataFrame
print('------------------')
print('------------------')
print(final_df)




In [ ]:
# Optionally, save to CSV
final_df.to_csv("class_counts_by_threshold.csv")

In [ ]:
final_df.loc[0.05].sum()

In [ ]:
final_df.loc[0.15].sum()

In [ ]:
final_df.loc[0.2].sum()

In [ ]:
final_df["RJ"].sort_index()

In [ ]:
# Define class labels
y_columns_2 = ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC", "RJ"]

# Dictionary to store final class counts per threshold
final_counts = {}

# Process each threshold file
read_chunk_size = 1000  # Adjust based on memory

threshold = 0.2
print(f"starting for threshold: {threshold}")
try:
    # File paths
    aug_file = f"./training_diff_thresholds_final/training_augmented_data_threshold_{threshold}.csv"
    orig_file = f"./training_diff_thresholds_final/training_data_threshold_{threshold}.csv"

    # Initialize class counts
    class_counts = {col: 0 for col in y_columns_2}

    # Read augmented data in chunks and accumulate counts
    for chunk in pd.read_csv(aug_file, chunksize=read_chunk_size):
        for col in y_columns_2:
            class_counts[col] += chunk[col].sum()

    # Read original (non-augmented) data in chunks and accumulate counts
    for chunk in pd.read_csv(orig_file, chunksize=read_chunk_size):
        for col in y_columns_2:
            class_counts[col] += chunk[col].sum()

    # Store in final dictionary
    final_counts[threshold] = class_counts

    print(threshold)
    print(class_counts)
    print('------------------')

except Exception as e:
    print(f"Skipping threshold {threshold} due to error: {e}")

# Convert to DataFrame
final_df = pd.DataFrame.from_dict(final_counts, orient="index")
final_df.index.name = "Threshold"

# Print the final DataFrame
print('------------------')
print('------------------')
print(final_df)


